# Module 9: AI-Assisted Exploration
### BWSI SGAI 2026

By the end of this module you'll know:
- What Claude Code is and how it reads your codebase
- The three effective uses: **Explain**, **Suggest extension**, **Debug**
- How to anchor Claude to the `.md` docs so it gives useful answers — not hallucinated ones
- How to frame a measurable research question for your team project

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## You Have AI Access — Now What?

You've now seen all 8 directories:

| Module | Directory | What you learned |
|--------|-----------|------------------|
| 1 | `gameplay/` | Game rules, scoring, constants |
| 2 | `data/` | Images, metadata CSV, class distribution |
| 3 | `ui/` | UI layout, color constants, clock modes |
| 4 | `game_interfaces/` | Wiring between `main.py`, env, and models |
| 5 | `models/` (perception) | CNN classifier, Oracle fallback, `DataParser` |
| 6 | `models/` (decision) | RL recommender, heuristic rules, DQN |
| 7 | `training/` | Training scripts, reward curves, baselines |
| 8 | `tests/` | Test coverage, adding assertions |

Claude Code is a tool that can read this codebase alongside you.  
This module teaches you to use it **effectively** — not just throw prompts at it.

INSTRUCTOR NOTES:
Ask students: "Has anyone used Claude or ChatGPT for coding before? What worked, what didn't?"
This primes the class for the 'what NOT to do' section. Common prior experience: asked it to write something big, got back code that didn't fit the project.
The key insight today: focused context = useful output. Unfocused context = hallucination.

## The Whole Codebase — With Its Maps

```
bwsi-sgai/
├── gameplay/
│   ├── humanoid.py
│   ├── scorekeeper.py
│   ├── constants.py
│   ├── enums.py
│   └── gameplay.md          ← Claude's map for this directory
├── data/
│   ├── consolidated_metadata.csv
│   └── data.md              ← Claude's map for this directory
├── ui/
│   └── ui.md                ← Claude's map
├── game_interfaces/
│   └── game_interfaces.md   ← Claude's map
├── models/
│   └── models.md            ← Claude's map
├── training/
│   └── training.md          ← Claude's map
├── tests/
│   └── tests.md             ← Claude's map
├── docs/
│   └── docs.md              ← Claude's map
└── main.py
```

*Every directory has a `.md` file that describes its files, classes, and extension points.  
These exist so that you — and Claude — can navigate the codebase without reading every line.*

## What Is Claude Code?

Claude Code is a **CLI tool** that runs in your terminal and has access to your files.

```bash
# Launch it from the repo root
claude
```

What it can do:
- **Read files** when you tell it where they are — it does not read the whole project automatically
- **Explain** code you point it at
- **Suggest changes** when you describe what you want
- **Debug** when you show it code and a symptom

What it **cannot** do:
- Run the game for you
- Know your codebase without being pointed at specific files
- Guarantee that its suggestions compile or pass tests — **you have to verify**

**You have a token allotment** — a budget for how much you can send to Claude.  
Use it on the hard parts: understanding tricky code, unblocking a bug, formulating your research question.  
Don't burn it asking Claude to write large features from scratch.

INSTRUCTOR NOTES:
Briefly demo launching 'claude' in the terminal and showing that it opens a REPL-style interface.
If Claude Code isn't installed on student machines, show a screenshot or the instructor's terminal.
Emphasize: tokens are finite. The question isn't 'can I ask Claude this' — it's 'is this the best use of my budget?'

## The .md Docs Are Claude's Anchor

```
bwsi-sgai/
├── gameplay/
│   └── gameplay.md          ← now
├── models/
│   └── models.md            ← now
├── game_interfaces/
│   └── game_interfaces.md
├── data/
│   └── data.md
├── training/
│   └── training.md
└── tests/
    └── tests.md
```

*Each `.md` doc contains: a directory diagram, a per-file table, and extension point sections.  
When you include one of these in your prompt, Claude gets the architecture in a structured format — no hallucination needed.*

*`bwsi-sgai/models/models.md`*

## The .md Docs — How to Use Them

The pattern is simple: **include the doc in your prompt.**

```
Read models/models.md and tell me: if I want to write a custom classifier
that always predicts State.HEALTHY, which class do I inherit from and
which method do I need to implement?
```

Why this works:
- You gave Claude the map for the directory
- You asked a specific, bounded question
- The answer Claude gives will match the actual ABC pattern in the code

Why the vague version fails:

```
How do I write a classifier for this game?    ← no file context, Claude guesses
```

*`bwsi-sgai/` — all `.md` reference docs*

## The .md Docs — Touch

Run the cell below to list every `.md` reference doc in the repo.  
These are the files you'll include in your Claude prompts.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
import os

docs = [
    os.path.join(root, f).replace("\\", "/")
    for root, _, files in os.walk(".")
    for f in files
    if f.endswith(".md") and ".ipynb_checkpoints" not in root
]
docs.sort()

print("Reference docs in this repo:")
for d in docs[:20]:  # first 20 — all the module docs are here
    print(" ", d)

Reference docs in this repo:
  ./.pytest_cache/README.md
  ./CLAUDE.md
  ./README.md
  ./data/data.md
  ./docs/architecture.md
  ./docs/button-state-machine.md
  ./docs/cheat-sheet.md
  ./docs/cooldown-activity.md
  ./docs/docs.md
  ./docs/game-design.md
  ./docs/instructor-guide.md
  ./docs/setup.md
  ./docs/student-README.md
  ./docs/troubleshooting.md
  ./docs/windows-setup.md
  ./game_interfaces/game_interfaces.md
  ./gameplay/gameplay.md
  ./lectures/lectures.md
  ./models/models.md
  ./tests/README.md


> Each of these files is a structured description of its directory's code.  
> When you're stuck on something in `gameplay/`, paste `gameplay/gameplay.md` into your Claude prompt before asking.

## Three Effective Uses

```
bwsi-sgai/
├── game_interfaces/
│   ├── data_parser.py       ← Use 1: Explain
│   └── game_interfaces.md
├── models/
│   ├── HeuristicRecommender.py  ← Use 3: Debug
│   └── models.md
└── data/
    └── data.md              ← Use 2: Suggest extension
```

We'll cover each use with a real prompt you can copy into Claude Code:  

1. **Explain** — understand code you didn't write  
2. **Suggest extension** — ask Claude what to change for a specific goal  
3. **Debug** — share code + symptom, ask what's wrong

*`bwsi-sgai/game_interfaces/data_parser.py`*

## Use 1: Explain

**The question:** You looked at `datarow_to_value()` in `data_parser.py` and you're not sure what it returns or how it decides the value.

**Good prompt:**

```
Read game_interfaces/data_parser.py and explain what datarow_to_value() does.
Specifically: what does it return, what fields from the CSV row does it use,
and what range of values should it produce?
```

**Why it works:**
- Gives Claude the exact file path — it reads the real code, not a guess
- Asks three specific sub-questions — Claude can't give a vague answer
- Scoped to one function — no wasted tokens on the rest of the file

**What to do with the answer:**  
Compare it to what you learned in Module 5. Does Claude's explanation match?  
If it doesn't, the `.md` doc is more authoritative than Claude's summary.

INSTRUCTOR NOTES:
The function signature is: datarow_to_value(data_row: pd.Series, rng: random.Random | None = None) -> float
It returns a float in [0.0, 1.0] representing triage priority.
It reads the 'State' and 'Item' columns from the CSV row.
Injured humanoids get higher values; zombies and corpses get lower values.
Students may find this surprising if they haven't looked at it closely — that's fine, it's a good 'explain' target.

*`bwsi-sgai/game_interfaces/data_parser.py` · `bwsi-sgai/data/data.md`*

## Use 2: Suggest Extension

**The goal:** You want `datarow_to_value()` to return a higher value for humanoids carrying a Bow (they're rare and useful).

**Good prompt:**

```
Read game_interfaces/data_parser.py and data/data.md.
I want datarow_to_value() to return 0.9 for humanoids whose Item column
is 'Bow' and 0.3 for all others, regardless of their State.
Write the change. Keep the function signature identical.
```

**Why it works:**
- Two files given — Claude sees both the function and the CSV column names
- The return value constraint is explicit: 0.9 and 0.3, not "higher" and "lower"
- Signature constraint mentioned — Claude won't add parameters you don't need

**Before running Claude's code — verify:**
1. The return type is still `float` in `[0.0, 1.0]`
2. The function signature matches what `DataParser` calls it with
3. No new imports were added that aren't available in the env

INSTRUCTOR NOTES:
The 'Item' column in consolidated_metadata.csv has values like 'Bow', 'Staff', 'Sword', 'None', etc.
The correct check in Python would be: if data_row['Item'] == 'Bow': return 0.9
Verify with students: how would they test this change without retraining? (Use the Touch cell pattern from Module 5:
  df = pd.read_csv('data/consolidated_metadata.csv'); row = df[df['Item']=='Bow'].iloc[0]; print(datarow_to_value(row))
Point out: this is a good research question setup — does changing value weights for Bow-carriers improve auto mode score?
The 'keep signature identical' instruction is critical — without it, Claude may add a parameter.

*`bwsi-sgai/models/HeuristicRecommender.py`*

## Use 3: Debug

**The symptom:** Your `HeuristicRecommender` always picks `SKIP` no matter what humanoid appears.

**Good prompt:**

````
My HeuristicRecommender always picks ActionState.SKIP regardless of
what the humanoid is. Here is my recommend() method:

```python
def recommend(self, humanoid, probs, scorekeeper):
    label, confidence = probs_to_class(probs)
    if label == State.ZOMBIE and confidence > 0.8:
        return ActionState.SQUISH
    if label == State.INJURED:
        return ActionState.SAVE
    return ActionState.SKIP
```

Probs is always [0.1, 0.7, 0.1, 0.1] when I print it.
What's wrong?
````

**Why it works:**
- Specific symptom: "always SKIP" — not "it doesn't work"
- Pastes only the changed method — not 200 lines of context
- Includes a concrete observation: what `probs` actually contains

**A second debug scenario — RL:**

```
I changed GAMMA from 0.99 to 0.9 in train_rl.py and retrained at 500 episodes.
The scores in summary.csv look identical to the baseline. Why?
Read training/train_rl.py and training/training.md.
```

**What Claude will check:**
- Did the change actually propagate to the `DQNAgent`?
- Did training actually run (`NUM_EPISODES=500` vs. early stopping)?
- Is `summary.csv` reading the new run or a cached result?

INSTRUCTOR NOTES:
The first debug example: probs = [0.1, 0.7, 0.1, 0.1] means probs[1] = 0.7 = HEALTHY (index 1).
Order is: probs[0]=zombie, probs[1]=healthy, probs[2]=injured, probs[3]=corpse.
So label would be State.HEALTHY, which hits the SKIP fallback. The bug is that the humanoid is healthy and the heuristic correctly skips.
The student's mistake is expecting INJURED behavior on a HEALTHY humanoid — not a code bug, a test-case bug.
This is a good teaching moment: Claude can only debug what you tell it. If you give it wrong probs values in your prompt, it may diagnose the wrong problem.

For the RL debug scenario: the most common real cause is that summary.csv appends across runs, so students are looking at an old row.
Tell students to sort by timestamp or clear summary.csv before comparing.

## What NOT to Do

**"Write me a new classifier that uses a different model architecture."**  
Claude will produce a class that doesn't inherit from `HumanoidClassifier` and doesn't implement `classify()`.  
It won't be injectable into `main.py`. You'll spend more time debugging the glue than writing the thing yourself.

---

**"Make the game score higher."**  
Vague goal = unconstrained changes. Claude might change constants you didn't intend to touch, or suggest retraining — without knowing your hardware limits.

---

**Pasting 500 lines and asking "what's wrong?"**  
Claude gets too much noise and too little signal about *what changed*.  
Instead: paste only the modified function + the symptom. Claude works with focused context.

---

**"Improve the RL training."**  
Claude will hallucinate hyperparameters that sound reasonable but weren't validated on this dataset.  
If you want to explore hyperparameters: read `training/training.md` first, then ask about one specific parameter at a time.

INSTRUCTOR NOTES:
If students push back ('but Claude is smart, it should handle vague prompts') — push back on that.
Claude is a text predictor. It predicts what a plausible answer looks like. It doesn't know your codebase constraints unless you tell it.
Analogy: asking a brilliant friend who's never seen your code to help debug it. They need the same context you'd give a human.
The constraint isn't intelligence — it's context. You are responsible for providing context.

## Your Research Question

Your team's goal: answer a measurable question about this codebase.

**Good format:**
> "Does changing **[X]** in **[file]** from **[value A]** to **[value B]** affect **[metric]** when measured with **[method]**?"

**Good example:**
> "Does setting `GAMMA=0.9` vs. `0.99` in `training/train_rl.py` affect the mean score in `auto` mode after 500 training episodes?"

**Bad examples:**
- "Can we make the game better?" — "better" is not measurable
- "What if we train longer?" — no comparison condition, no target metric
- "Can the CNN be improved?" — no controlled variable

**Claude can help you:**
- Identify which files to change for your variable of interest
- What to hold constant while you vary one thing
- How to read the output (`summary.csv`, reward curves, action counts)

**Comparison trap:**  
If comparing two conditions, keep episode count the same.  
A 500-episode policy vs. a 2500-episode policy is comparing training time, not your variable.

INSTRUCTOR NOTES:
Give teams 5 minutes to draft their research question in the format above.
Have each team read it aloud. Ask the class: is the variable controlled? Is the metric measurable?
Common draft: 'We want to see if a better CNN helps.' Push: better by what measure? Accuracy? Score in auto mode? What's the baseline?
Prompt Claude usage here: 'Give Claude your draft question and ask it to identify which files you need to change and what to hold constant.'
Remind: you have a token allotment. This is a good use of it.

## Exercise: Three Prompts, Three Outcomes

This exercise runs entirely inside Claude Code — no code cell needed.  
Open a terminal, launch `claude` from the repo root, and run all three parts.

**Time: ~15 minutes total**

---

**Part 1 — Explain (5 min)**

Copy this prompt into Claude Code:

```
Read game_interfaces/data_parser.py and explain what datarow_to_value() does.
Specifically: what does it return, what fields from the CSV row does it use,
and what range of values should it produce?
```

Does Claude's explanation match what you learned in Module 5? Note any differences.

## Exercise: Three Prompts, Three Outcomes (cont.)

**Part 2 — Suggest extension (5 min)**

Copy this prompt into Claude Code:

```
Read game_interfaces/data_parser.py and data/data.md.
I want datarow_to_value() to return 0.9 for humanoids whose Item column
is 'Bow' and 0.3 for all others, regardless of their State.
Write the change. Keep the function signature identical.
```

Before running the suggested code, check:
- Is the return type still `float` in `[0.0, 1.0]`?
- Does the function signature match the original?
- Did Claude add any imports?

You don't have to apply the change — just review whether the suggestion fits the pattern.

---

**Part 3 — Debug (5 min)**

1. Open `models/HeuristicRecommender.py`
2. Find the `recommend()` method and add `return ActionState.SKIP` as the very first line (before any logic)
3. Copy this prompt into Claude Code:

```
My HeuristicRecommender.recommend() always returns ActionState.SKIP.
Read models/HeuristicRecommender.py. What's wrong?
```

Did Claude find the unconditional early return? Undo the change after.

INSTRUCTOR NOTES:
Part 1: Students may find Claude's explanation matches perfectly, or adds detail they didn't get from Module 5 alone. Both are fine — point out that Claude read the actual source, not a summary.
Part 2: Claude should produce something like: if data_row.get('Item') == 'Bow': return 0.9 / return 0.3. The key check is signature preservation. If Claude adds a parameter, that's a bad suggestion — point out why (DataParser calls it with a fixed signature).
Part 3: Claude Code can read the file. It will find the early return immediately because it reads line by line. The debugging prompt works because it's specific — 'always SKIP' and 'read this file' is enough.
Make sure students undo the deliberate bug before moving on.

## Pop Quiz — Strategic Thinking

These are strategy questions, not recall.

---

**Q1.** You want Claude to explain how `_make_components()` works in `main.py`.  
What context should you include in your prompt?

A) The entire `main.py` file  
B) The file path and method name and what you're trying to understand  
C) A description of what you think the method does  
D) Nothing — Claude knows the file from previous prompts

**Answer: B**  
Tell Claude: the file path (`main.py`), the method name (`_make_components`), and what you're trying to understand or change.  
Option A wastes tokens on code Claude doesn't need. Option D is wrong — Claude has no memory between sessions, and even within a session it doesn't automatically read files.

## Pop Quiz — Strategic Thinking

**Q2.** Claude suggests a change to `ActionRecommender.recommend()`.  
Before running it, what do you check?

A) That the code has no syntax errors  
B) That the return type is `ActionState`, the method signature matches the ABC, and no new imports are needed  
C) That Claude's confidence was high in its response  
D) Run it and see if the tests pass

**Answer: B**  
`ActionRecommender` is an abstract base class — any method override must return `ActionState` and match the signature exactly. If the return type or signature diverges, the game will crash at runtime, not at import time.  
Option D isn't wrong as a final step, but checking the contract first saves you a debug cycle.

## Pop Quiz — Strategic Thinking

**Q3.** Your team's research question is:  
> "Does the game get better if we train longer?"

What's wrong with it?

A) Nothing — it's a valid research question  
B) "Better" is undefined, there's no controlled variable, and there's no comparison condition  
C) You can't change training length in this codebase  
D) The question is too narrow

**Answer: B**  
"Better" is not measurable. There's no comparison condition (train longer than *what*?).  
Reframe: **"Does increasing `NUM_EPISODES` from 500 to 2500 in `training/train_rl.py` increase the mean score in `auto` mode when both conditions use the same `SEED` and `GAMMA`?"**  
Now it's testable: two runs, one variable, one metric, one measurement method.

INSTRUCTOR NOTES:
Q3 is the most important question in the module. Students will draft bad research questions if this framing isn't clear.
Walk through the reframed version word by word: NUM_EPISODES (variable), 500 vs 2500 (values), auto mode (game mode), mean score (metric), same SEED and GAMMA (controls).
Point out that Claude can help them reframe — 'Here's my draft question: [question]. Is it measurable? What would I need to control?'

## You've Completed All 9 Modules

You know where everything lives:

| What you want to change | Where to look |
|------------------------|---------------|
| Game rules, weights, timing | `gameplay/constants.py`, `gameplay/enums.py` |
| Triage value assignment | `game_interfaces/data_parser.py` |
| Heuristic decision rules | `models/HeuristicRecommender.py` |
| RL training hyperparameters | `training/train_rl.py`, `training/train_rl.ipynb` |
| CNN training | `training/train_cnn.py`, `training/train_cnn.ipynb` |
| All tunable parameters | `gameplay/constants.py` |
| Architecture overview | `docs/architecture.md` |

**For AI-assisted work:**
- Always give Claude a file path and a specific question
- Include the relevant `.md` doc for context
- Verify Claude's suggestions against the ABC contracts before running
- Use your token budget on the hard parts

---

*Now go answer a research question.*